In [ ]:
import json, re, pandas as pd, numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from Run_Pipeline.Env_Loader import EnvLoader

# --------- Resume ---------
resume = {
    "자기소개": "저는 AI 기반의 채용 공고 분석 및 추천 시스템을 개발하는 엔지니어입니다.",
    "경력": "5년 이상의 소프트웨어 엔지니어 경력, AI 및 데이터 분석 프로젝트 수행.",
    "기술 스택": "Python, TensorFlow, PyTorch, HuggingFace, LangChain, ChromaDB",
    "프로젝트 경험": "AI 채용 공고 분석 시스템 개발 등"
}
resume_text = " ".join(resume.values())

# Extract basic skill set
resume_skills = {s.strip().lower() for s in resume["기술 스택"].split(",")}

# --------- Two simplified JDs ---------
jds = {
    "283170": {
        "제목": "프로모션 기획 마케터",
        "요구 최소 경력": 3,
        "본문": "Amplitude 활용 가능, 이커머스 프로모션 경험, 데이터 기반 분석, 마케팅 커뮤니케이션"
    },
    "282132": {
        "제목": "콘텐츠 에디터",
        "요구 최소 경력": 0,
        "본문": "콘텐츠 기획, 카피라이팅, Figma, Notion, 온라인 커머스 경험 우대"
    }
}

rows = []
for jid, d in jds.items():
    rows.append({"jd_id": jid, "제목": d["제목"], "요구 최소 경력": d["요구 최소 경력"], "full_text": d["본문"]})
jd_df = pd.DataFrame(rows)

# --------- Text similarity (TF-IDF cosine) ---------
docs = [resume_text] + jd_df["full_text"].tolist()
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(docs)
cosine_scores = cosine_similarity(X[0:1], X[1:]).flatten()
jd_df["embedding_sim"] = cosine_scores


# --------- Skill coverage ---------
def extract_words(text): return {w.lower() for w in re.findall(r"[A-Za-z+#]{2,}", text)}


jd_df["skill_hits"] = jd_df["full_text"].apply(lambda t: len(resume_skills & extract_words(t)))
jd_df["coverage"] = jd_df["skill_hits"] / (len(resume_skills) or 1)

# --------- Experience penalty ---------
resume_years = 5
jd_df["exp_penalty"] = jd_df["요구 최소 경력"].apply(lambda miny: 1.0 if resume_years >= miny else resume_years / (miny or 1))

# --------- Final score ---------
jd_df["final_score"] = 0.6 * jd_df["embedding_sim"] + 0.3 * jd_df["coverage"] + 0.1 * jd_df["exp_penalty"]
out = jd_df[["jd_id", "제목", "embedding_sim", "coverage",
             "exp_penalty", "final_score"]].sort_values("final_score", ascending=False)

print(out.to_markdown(index=False))

In [ ]:
import json

# 사용자 이력서 데이터
resume_data = {
    "자기소개": "저는 AI 기반의 채용 공고 분석 및 추천 시스템을 개발하는 엔지니어입니다. 다양한 기술 스택과 직군에 대한 이해를 바탕으로, 사용자 맞춤형 채용 정보를 제공하고 있습니다.",
    "경력": "저는 5년 이상의 경력을 가진 소프트웨어 엔지니어로, AI 및 데이터 분석 분야에서 다양한 프로젝트를 수행해왔습니다. 특히, 채용 공고 분석 및 추천 시스템 개발에 집중하고 있습니다.",
    "기술 스택": "Python, TensorFlow, PyTorch, HuggingFace Transformers, LangChain, ChromaDB",
    "프로젝트 경험": "AI 기반 채용 공고 분석 시스템 개발, 사용자 맞춤형 채용 정보 추천 시스템 구축",
    "수상 경력": "2023년 AI 개발자 대회 우수상 수상, 2022년 데이터 분석 경진대회 최우수상 수상",
    "기타": "AI 및 데이터 분석 관련 블로그 운영, 오픈소스 프로젝트 기여"
}

# 채용 공고 데이터 (예시로 하나만 사용)
job_post_data = {
    "283170": {
        "제목": "프로모션 기획 마케터",
        "회사 소개": "컬리 온사이트 마케팅 조직은 고객이 컬리에서 즐거운 장보기 경험을 하도록 돕고 그에 따라 고객들이 더 활발한 구매를 하게 만드는 것을 목표로 합니다 고객의 니즈와 상품의 특성을 파악하고 분석하여, 매력적인 혜택이나 브랜드, 상품이 고객의 눈에 띌 수 있도록 큐레이션 하고 노출합니다 프로모션 기획 마케터는 온사이트 프로모션을 계획 및 실행함으로써 컬리에 방문한 유저의 즐거운 장보기 경험과 팀 목표달성에 기여합니다 고객의 니즈와 시장의 변화를 캐치하여 적절한 프로모션 스킴을 기획하고 긍정적인 고객 경험을 위해 책임감을 갖고 임합니다",
        "주요 업무": ["사업 목표 달성을 위한 온사이트 프로모션 전략 수립 및 실행", "", "1 연분기월 별 온사이트 운영 전략 모색",
                  " 전사와 담당 카테고리 관점의 프로모션 계획을 수립하고 실행합니다", " 월 단위의 주요 프로모션 테마를 기획합니다", "", "2 온사이트 프로모션이벤트 기획 및 운영 ",
                  " 프로모션 운영에 필요한 일정 및 운영 프로세스를 기획합니다", " 상세 페이지를 포함해 고객이 프로모션을 만날 수 있는 모든 지면을 운영합니다", "",
                  "3 프로모션 성과 분석을 통한 개선안 도출 ", " 프로모션의 성과가 영향을 미칠 수 있는 전 영역에 대해 데이터 기반으로 분석합니다",
                  " 쿠폰 및 할인 등의 실제예상 비용 지표를 관리합니다"],
        "자격 요건": ["경험", " 최소 1년 이상의 유관 경험 또는 그에 준하는 역량을 보유하신 분", " 경력의 절반 이상 이커머스 산업 재직 경험이 있으신 분",
                  " 이커머스 MD 또는 마케팅 경험이 1년 이상 있으신 분 ", " 프로모션을 통해 매출을 개선한 경험을 보유하신 분", "", "지식",
                  " 커머스에 대한 높은 이해도를 갖추신 분", "", "스킬능력", " 유관 부서와 원활하고 명확한 커뮤니케이션 역량이 있으신 분",
                  " 데이터 기반으로 한 정량적인 근거 제시가 가능하신 분", " 꼼꼼한 업무처리 능력을 보유하신 분"],
        "우대 사항": ["Amplitude 활용이 가능하신 분", " 서비스 마케팅, 사업 전략까지 확장 가능한 역량을 보유하신 분",
                  " 프로모션에 필요한 기획서 작성 eg 카피라이팅에 뛰어난 능력을 가지신 분", " IT, 이커머스, 마케팅, 라이프스타일의 최신 트렌드에 밝으신 분"],
        "복지": ["모두가 집중하고 성장할 수 있도록, 최적의 근무 환경을 제공해요", " ", " 리프레시도 필요하죠, 매월 유급 반차05일 퍼플데이 제공",
               " 장기근속에 대한 유급휴가 및 휴가비 지원제도 운영", "", "", "건강하고 활력 넘치는 직장생활을 위해 컬리가 지원해요", "",
               " 여러분이 가장 소중한 고객이니까, 컬리 적립금 지급", " 건강한 신체만큼 중요한 건 없죠, 의료비 지원", " 허리 건강을 위한 모션 데스크 기본 지원",
               " 삶과 업무에 지쳐 멘탈 케어가 필요하다면, 프라이빗 심리 상담 지원"],
        "채용절차": ["합류 여정", "지원서 제출 직무 적합성 인터뷰\xa0 직무적합성검사 및 종합 인터뷰\xa0 최종 합격", "", "꼭 확인해주세요",
                 " 종합 인터뷰 전, 온라인 직무적합성검사가 진행됩니다", " 세부 절차는 서류 합격자 분들께 개별 안내드릴 예정입니다",
                 " 프로세스는 일정과 상황에 따라 사전 안내 후 일부 변경될 수 있으며, 각 전형의 결과는 개별 안내드립니다 ", "", "기타 사항", " 고용형태는 정규직수습 3개월 입니다",
                 " 국가 유공자, 장애인 등 취업보호대상자는 관계 법령에 따라 우대합니다", " 해외 여행에 결격 사유가 없어야 하며, 남성은 군필 혹은 면제자여야 합니다",
                 " 지원서에 기재된 내용 중 허위 사실이 발견될 경우 채용이 취소될 수 있습니다"],
        "장점": ["의료비지원", "휴가비", "유급휴가", "커피스낵바", "1,00110,000명", "설립10년이상", "유망산업", "보너스", "연봉 업계평균이상", "적극 채용 중",
               "AI 선도 기업", "누적투자100억이상"],
        "회사명": "컬리",
        "근무지": "서울",
        "지역": "강남구",
        "직군": "마케팅광고",
        "직무": ["마케터", "디지털 마케터", "모바일마케팅"],
        "기술 스택": [""],
        "요구 최소 경력": 3,
        "요구 최대 경력": 10,
        "신입 지원 가능 여부": False,
        "마감일": "상시 공고"
    }
}


# 텍스트 결합 함수 정의
def combine_resume_text(resume):
    combined_text = ""
    combined_text += f"자기소개: {resume.get('자기소개', '')}. "
    combined_text += f"경력: {resume.get('경력', '')}. "
    combined_text += f"기술 스택: {resume.get('기술 스택', '')}. "
    combined_text += f"프로젝트 경험: {resume.get('프로젝트 경험', '')}. "
    combined_text += f"수상 경력: {resume.get('수상 경력', '')}. "
    combined_text += f"기타: {resume.get('기타', '')}."
    return combined_text.strip()


def combine_job_post_text(job_post):
    combined_text = ""
    combined_text += f"제목: {job_post.get('제목', '')}. "
    combined_text += f"회사 소개: {job_post.get('회사 소개', '')}. "
    combined_text += f"주요 업무: {' '.join(job_post.get('주요 업무', []))}. "
    combined_text += f"자격 요건: {' '.join(job_post.get('자격 요건', []))}. "
    combined_text += f"우대 사항: {' '.join(job_post.get('우대 사항', []))}. "
    # 경력 정보는 숫자로 비교할 것이므로 텍스트에 포함하지 않음
    return combined_text.strip()


# 텍스트 결합
resume_text = combine_resume_text(resume_data)
job_post_id = list(job_post_data.keys())[0]  # 첫 번째 공고 ID 가져오기
job_post_text = combine_job_post_text(job_post_data[job_post_id])

print("--- 결합된 이력서 텍스트 ---")
print(resume_text)
print("\n--- 결합된 채용 공고 텍스트 ---")
print(job_post_text)

In [ ]:
from sentence_transformers import SentenceTransformer, util
import torch

# 한국어에 특화된 SBERT 모델 로드 (KoBERT 기반)
# 'snunlp/KR-SBERT-V40K-etri-distilbert' 또는 'jhgan/ko-sroberta-multitask' 등을 사용할 수 있습니다.
# 여기서는 'jhgan/ko-sroberta-multitask' 모델을 사용합니다.
model_name = 'jhgan/ko-sroberta-multitask'
model = SentenceTransformer(model_name)

# 텍스트를 임베딩으로 변환
resume_embedding = model.encode(resume_text, convert_to_tensor=True)
job_post_embedding = model.encode(job_post_text, convert_to_tensor=True)

print(f"\n이력서 임베딩 차원: {resume_embedding.shape}")
print(f"채용 공고 임베딩 차원: {job_post_embedding.shape}")

In [ ]:
# 코사인 유사도 계산
cosine_similarity_score = util.cos_sim(resume_embedding, job_post_embedding).item()

print(f"\n의미적 유사도 (코사인 유사도): {cosine_similarity_score:.4f}")

In [ ]:
# 1. 경력 조건 검토
# 채용 공고의 요구 경력 범위
required_min_experience = job_post_data[job_post_id].get("요구 최소 경력", 0)
required_max_experience = job_post_data[job_post_id].get("요구 최대 경력", 999)  # 최대 경력이 없으면 아주 크게 설정

# 이력서에서 경력 기간 추출 (간단한 예시, 실제로는 더 복잡한 파싱 필요)
# 이력서의 '경력' 필드에서 숫자와 '년'을 찾아 경력 기간으로 간주
import re

resume_experience_match = re.search(r'(\d+)\s*년', resume_data.get('경력', ''))
resume_experience = int(resume_experience_match.group(1)) if resume_experience_match else 0

experience_match_score = 0
if required_min_experience <= resume_experience <= required_max_experience:
    experience_match_score = 1.0  # 경력 조건 충족 시 1점
elif resume_experience > required_max_experience:
    experience_match_score = 0.5  # 경력이 너무 많아도 감점 (Overqualified)
else:
    experience_match_score = 0.0  # 경력 미달 시 0점

print(f"\n이력서 경력: {resume_experience}년")
print(f"공고 요구 경력: {required_min_experience}~{required_max_experience}년")
print(f"경력 매칭 점수: {experience_match_score:.2f}")

# 2. 기술 스택 매칭 (간단한 예시)
job_tech_stacks = [ts.strip().lower() for ts in job_post_data[job_post_id].get("기술 스택", []) if ts.strip()]
resume_tech_stacks = [ts.strip().lower() for ts in resume_data.get("기술 스택", "").split(',') if ts.strip()]

# 공고에 기술 스택이 비어있는 경우 (일부 공고는 명시하지 않음)
if not job_tech_stacks:
    tech_stack_score = 0.5  # 기술 스택 미명시 시 중간 점수
else:
    matched_tech_count = sum(1 for tech in resume_tech_stacks if tech in job_tech_stacks)
    tech_stack_score = matched_tech_count / len(job_tech_stacks) if job_tech_stacks else 0

print(f"이력서 기술 스택: {resume_tech_stacks}")
print(f"공고 기술 스택: {job_tech_stacks}")
print(f"기술 스택 매칭 점수: {tech_stack_score:.2f}")

# 3. 직군 매칭 (AI 엔지니어 vs 마케터)
job_직군 = job_post_data[job_post_id].get("직군", "").lower()
# 이력서 내용에서 "엔지니어", "개발자" 키워드를 통해 직군 유추
resume_직군_keywords = ["엔지니어", "개발자", "ai", "데이터 분석"]
resume_직군_match = any(keyword in resume_text.lower() for keyword in resume_직군_keywords)

직군_match_score = 0.0
if "마케팅" in job_직군 and not resume_직군_match:
    직군_match_score = 0.0  # 마케팅 직군인데 개발자 이력서면 0점
elif "마케팅" not in job_직군 and resume_직군_match:
    직군_match_score = 0.5  # 마케팅 직군이 아니면서 개발자 이력서면 어딘가 맞을수도 있으니 0.5점 (이상적인 매칭은 아님)
else:
    # 이 부분은 직군 키워드가 더 정확히 매핑되어야 합니다.
    # 이 예시에서는 직무가 매우 달라 낮은 점수 부여
    직군_match_score = 0.0  # 직군 불일치 시 0점 (예: 개발 이력서가 마케팅 공고에 지원)

print(f"공고 직군: {job_직군}")
print(f"이력서 직군 관련 키워드 매칭: {resume_직군_match}")
print(f"직군 매칭 점수: {직군_match_score:.2f}")

In [ ]:
# 가중치 설정 (조절 가능)
weight_cosine = 0.6  # 의미적 유사도에 가장 높은 가중치
weight_experience = 0.2  # 경력 매칭
weight_tech_stack = 0.1  # 기술 스택 매칭
weight_job_role = 0.1  # 직군 매칭 (매우 중요)

# 최종 적합도 점수 계산
final_적합도_score = (
        cosine_similarity_score * weight_cosine +
        experience_match_score * weight_experience +
        tech_stack_score * weight_tech_stack +
        직군_match_score * weight_job_role
)

print(f"\n--- 최종 적합도 점수 ---")
print(f"코사인 유사도 가중치: {weight_cosine}")
print(f"경력 매칭 가중치: {weight_experience}")
print(f"기술 스택 매칭 가중치: {weight_tech_stack}")
print(f"직군 매칭 가중치: {weight_job_role}")
print(f"최종 적합도 점수: {final_적합도_score:.4f}")

# 결과 해석
if final_적합도_score >= 0.7:
    print("-> 매우 높은 적합도입니다.")
elif final_적합도_score >= 0.5:
    print("-> 높은 적합도입니다.")
elif final_적합도_score >= 0.3:
    print("-> 보통 적합도입니다.")
else:
    print("-> 낮은 적합도입니다.")

In [ ]:
import json  # JSON 파일 처리를 위한 라이브러리
import re  # 정규 표현식 (텍스트 패턴 검색)을 위한 라이브러리
from sentence_transformers import SentenceTransformer, util  # Sentence-BERT 모델 및 유틸리티
import torch  # PyTorch 기반의 텐서 연산을 위한 라이브러리 (SBERT 내부적으로 사용)
import os  # 파일 경로 등을 다루기 위한 라이브러리

# --- 설정 ---
JOB_POST_FILE = '../Data_Files/wanted_detail_improve_20250604.json'  # 채용 공고 파일 경로
RESUME_FILE = '../Data_Files/resume.json'  # 이력서 파일 경로
SBERT_MODEL_NAME = 'jhgan/ko-sroberta-multitask'  # 한국어에 최적화된 SBERT 모델 이름

# 최종 적합도 점수 계산에 사용될 각 요소의 가중치 (이 값을 조정하여 중요도를 바꿀 수 있습니다)
WEIGHT_COSINE_SIMILARITY = 0.6  # 의미적 유사성의 중요도
WEIGHT_EXPERIENCE_MATCH = 0.2  # 경력 일치 여부의 중요도
WEIGHT_TECH_STACK_MATCH = 0.15  # 기술 스택 일치 여부의 중요도
WEIGHT_JOB_ROLE_MATCH = 0.05  # 직군 일치 여부의 중요도 (이 예시에서는 AI 개발자 이력서와 마케터 공고이므로 낮게 설정)

# --- 1. 데이터 불러오기 ---
try:
    with open(RESUME_FILE, 'r', encoding='utf-8') as f:
        resume_data = json.load(f)[0]  # resume.json이 리스트 형태이고 첫 번째 요소가 이력서라고 가정
except FileNotFoundError:
    print(f"오류: {RESUME_FILE} 파일을 찾을 수 없습니다. 파일 경로를 확인해 주세요.")
    exit()
except json.JSONDecodeError:
    print(f"오류: {RESUME_FILE} 파일의 JSON 형식이 올바르지 않습니다.")
    exit()

try:
    with open(JOB_POST_FILE, 'r', encoding='utf-8') as f:
        job_post_data = json.load(f)
except FileNotFoundError:
    print(f"오류: {JOB_POST_FILE} 파일을 찾을 수 없습니다. 파일 경로를 확인해 주세요.")
    exit()
except json.JSONDecodeError:
    print(f"오류: {JOB_POST_FILE} 파일의 JSON 형식이 올바르지 않습니다.")
    exit()

print(f"{JOB_POST_FILE}에서 {len(job_post_data)}개의 채용 공고를 불러왔습니다.")
print(f"{RESUME_FILE}에서 이력서를 불러왔습니다.")


# --- 2. 이력서 전처리 ---
def combine_resume_text(resume):
    # 이력서의 여러 필드를 하나의 문자열로 합칩니다.
    combined_text = ""
    combined_text += f"자기소개: {resume.get('자기소개', '')}. "
    combined_text += f"경력: {resume.get('경력', '')}. "
    combined_text += f"기술 스택: {resume.get('기술 스택', '')}. "
    combined_text += f"프로젝트 경험: {resume.get('프로젝트 경험', '')}. "
    combined_text += f"수상 경력: {resume.get('수상 경력', '')}. "
    combined_text += f"기타: {resume.get('기타', '')}."
    return combined_text.strip()  # 양쪽 공백 제거


# 이력서 텍스트 합치기
resume_combined_text = combine_resume_text(resume_data)

# 이력서에서 경력 기간 추출 (정규표현식으로 숫자와 '년'을 찾습니다)
resume_experience_match = re.search(r'(\d+)\s*년', resume_data.get('경력', ''))
RESUME_EXPERIENCE_YEARS = int(resume_experience_match.group(1)) if resume_experience_match else 0

# 이력서의 기술 스택을 소문자 리스트로 변환합니다.
RESUME_TECH_STACKS = [ts.strip().lower() for ts in resume_data.get("기술 스택", "").split(',') if ts.strip()]

# 이력서의 직무를 나타내는 키워드 (AI/데이터/소프트웨어 엔지니어 관련)
RESUME_JOB_ROLE_KEYWORDS = ["ai", "데이터 분석", "엔지니어", "개발자", "소프트웨어"]

print("\n--- 전처리된 이력서 ---")
print(f"합쳐진 텍스트: {resume_combined_text[:100]}...")  # 처음 100자만 출력
print(f"경력 (년): {RESUME_EXPERIENCE_YEARS}")
print(f"기술 스택: {RESUME_TECH_STACKS}")


# --- 3. 채용 공고 전처리 함수 ---
def combine_job_post_text(job_post):
    # 채용 공고의 여러 필드를 하나의 문자열로 합칩니다.
    combined_text = ""
    combined_text += f"제목: {job_post.get('제목', '')}. "
    combined_text += f"회사 소개: {job_post.get('회사 소개', '')}. "
    # 리스트 형태의 필드는 공백으로 연결하고, 빈 문자열은 제외합니다.
    combined_text += f"주요 업무: {' '.join(filter(None, job_post.get('주요 업무', [])))}. "
    combined_text += f"자격 요건: {' '.join(filter(None, job_post.get('자격 요건', [])))}. "
    combined_text += f"우대 사항: {' '.join(filter(None, job_post.get('우대 사항', [])))}. "
    return combined_text.strip()


# --- 4. 임베딩 생성 ---
print(f"\nSentence-BERT 모델 로드 중: {SBERT_MODEL_NAME}...")
model = SentenceTransformer(SBERT_MODEL_NAME)  # 모델 로드
print("모델 로드 완료.")

# 이력서 임베딩 생성
resume_embedding = model.encode(resume_combined_text, convert_to_tensor=True)

# --- 5. 각 채용 공고에 대한 적합도 점수 계산 ---
best_match_job_id = None  # 가장 높은 점수를 받은 채용 공고 ID
highest_score = -1.0  # 최고 점수
job_post_scores = {}  # 각 채용 공고의 점수를 저장할 딕셔너리

print("\n모든 채용 공고에 대한 적합도 점수 계산 중...")

# 모든 채용 공고를 순회하며 점수 계산
for job_id, job_post in job_post_data.items():
    job_post_combined_text = combine_job_post_text(job_post)
    job_post_embedding = model.encode(job_post_combined_text, convert_to_tensor=True)

    # 1. 코사인 유사도 계산 (의미적 적합도)
    cosine_similarity_score = util.cos_sim(resume_embedding, job_post_embedding).item()

    # 2. 경력 매칭 점수
    required_min_exp = job_post.get("요구 최소 경력", 0)  # 최소 경력 (없으면 0년)
    required_max_exp = job_post.get("요구 최대 경력", 999)  # 최대 경력 (없으면 매우 큰 값)

    experience_match_score = 0.0
    if required_min_exp <= RESUME_EXPERIENCE_YEARS <= required_max_exp:
        experience_match_score = 1.0  # 경력 조건 완벽 충족
    elif RESUME_EXPERIENCE_YEARS > required_max_exp:
        experience_match_score = 0.5  # 경력이 너무 많음 (오버 스펙) - 점수는 조정 가능
    elif RESUME_EXPERIENCE_YEARS < required_min_exp:
        experience_match_score = 0.0  # 경력 미달

    # 3. 기술 스택 매칭 점수
    job_tech_stacks = [ts.strip().lower() for ts in job_post.get("기술 스택", []) if ts.strip()]
    tech_stack_score = 0.0
    if job_tech_stacks:  # 채용 공고에 기술 스택이 명시된 경우
        matched_tech_count = sum(1 for tech in RESUME_TECH_STACKS if tech in job_tech_stacks)
        tech_stack_score = matched_tech_count / len(job_tech_stacks)  # 매칭된 비율
    else:  # 채용 공고에 기술 스택이 명시되지 않은 경우 (중립적인 점수 부여)
        tech_stack_score = 0.5  # 이 부분은 필요에 따라 0으로 하거나 다르게 설정 가능

    # 4. 직군 매칭 점수
    job_직군 = job_post.get("직군", "").lower()
    직무_list = [d.lower() for d in job_post.get("직무", []) if d.strip()]  # 구체적인 직무 리스트

    job_role_match_score = 0.0
    # 이력서의 직무 키워드가 공고의 직군 또는 직무 리스트에 포함되는지 확인
    if any(keyword in job_직군 for keyword in RESUME_JOB_ROLE_KEYWORDS) or \
            any(any(keyword in specific_job for keyword in RESUME_JOB_ROLE_KEYWORDS) for specific_job in 직무_list):
        job_role_match_score = 1.0  # 직군이 일치 (예: 이력서-개발, 공고-IT개발)
    elif "마케팅" in job_직군 or "광고" in job_직군 or "기획" in job_직군:
        # 이 예시 이력서는 AI 개발자이므로, 마케팅/광고/기획 직군은 불일치로 간주
        job_role_match_score = 0.0
    else:  # 그 외의 직군 (부분적으로 유사하거나 판단이 어려운 경우)
        job_role_match_score = 0.3  # 이 점수도 조정 가능 (예: 영업 직군 등)

    # 최종 가중치 합산 점수 계산
    current_score = (
            cosine_similarity_score * WEIGHT_COSINE_SIMILARITY +
            experience_match_score * WEIGHT_EXPERIENCE_MATCH +
            tech_stack_score * WEIGHT_TECH_STACK_MATCH +
            job_role_match_score * WEIGHT_JOB_ROLE_MATCH
    )

    # 각 공고의 점수와 상세 내역 저장
    job_post_scores[job_id] = {
        "title": job_post.get("제목", "제목 없음"),
        "score": current_score,
        "details": {
            "cosine_similarity": cosine_similarity_score,
            "experience_match": experience_match_score,
            "tech_stack_match": tech_stack_score,
            "job_role_match": job_role_match_score
        }
    }

    # 최고 점수 갱신
    if current_score > highest_score:
        highest_score = current_score
        best_match_job_id = job_id

# --- 6. 결과 출력 및 가장 적합한 공고 표시 ---
print("\n--- 가장 적합한 채용 공고 (상위 10개) ---")
# 점수를 기준으로 채용 공고들을 내림차순 정렬
sorted_job_posts = sorted(job_post_scores.items(), key=lambda item: item[1]['score'], reverse=True)

# 상위 10개 공고 정보 출력
for i, (job_id, data) in enumerate(sorted_job_posts[:10]):
    print(f"\n{i + 1}. 공고 ID: {job_id}")
    print(f"   제목: {data['title']}")
    print(f"   종합 적합도 점수: {data['score']:.4f}")
    print(f"   상세 점수:")
    for detail_key, detail_value in data['details'].items():
        print(f"     - {detail_key.replace('_', ' ').title()}: {detail_value:.4f}")  # 언더바를 공백으로 바꾸고 첫 글자를 대문자로

if best_match_job_id:
    print(f"\n--- 최종 베스트 매칭 채용 공고 ---")
    best_match_data = job_post_scores[best_match_job_id]
    print(f"공고 ID: {best_match_job_id}")
    print(f"제목: {best_match_data['title']}")
    print(f"종합 적합도 점수: {best_match_data['score']:.4f}")
    print(f"상세 점수:")
    for detail_key, detail_value in best_match_data['details'].items():
        print(f"  - {detail_key.replace('_', ' ').title()}: {detail_value:.4f}")
else:
    print("\n처리할 채용 공고를 찾을 수 없습니다.")

In [ ]:
import json, csv, random
from pathlib import Path
import os

BASE_DIR = Path(os.getcwd()).parent
JOB_FILE = BASE_DIR / DATA_DIR / "wanted_detail_improve_20250604.json"
RESUME_FILE = BASE_DIR / DATA_DIR / "resume_sample.json"
OUT_CSV = BASE_DIR / DATA_DIR / "resume_job_pairs.csv"

# ── 데이터 로드 ─────────────────────────────────────────
jobs = json.loads(Path(JOB_FILE).read_text(encoding="utf-8"))
resumes = json.loads(Path(RESUME_FILE).read_text(encoding="utf-8"))


# ── 이력서 / 공고를 한 줄 텍스트로 합치는 헬퍼 ──────────
def combine_resume(r: dict) -> str:
    return (
        f"{r['university']} {r['major']} {r['education_status']} GPA {r['gpa']}. "
        f"경력년수: {r.get('experience_years', 0)}년. "
        f"희망직무: {', '.join(r['job_interests'])}. "
        f"기술: {r['skills']}."
    )


def combine_job(j: dict) -> str:
    return (
        f"제목: {j['제목']}. "
        f"주요 업무: {' '.join(j['주요 업무'])}. "
        f"자격 요건: {' '.join(j['자격 요건'])}. "
        f"우대: {' '.join(j['우대 사항'])}. "
        f"경력: {j.get('요구 최소 경력', 0)}~{j.get('요구 최대 경력', 99)}년."
    )


# ── (예) 이력서 1개 × 무작위 공고 300개 샘플 ────────────
pairs = []
N_SAMPLE = 300
sampled_jobs = random.sample(list(jobs.items()), k=N_SAMPLE)

for res in resumes[:1]:  # 하나의 이력서만 예시
    for job_id, job in sampled_jobs:
        pairs.append({
            "resume_text": combine_resume(res),
            "job_text": combine_job(job),
            "score": ""  # 빈칸 → 사람이 채점
        })

# ── CSV 저장 ────────────────────────────────────────────
with OUT_CSV.open("w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=["resume_text", "job_text", "score"])
    w.writeheader();
    w.writerows(pairs)

print(f"✅  CSV 저장 완료 → {OUT_CSV}")


In [ ]:
import os

BASE_DIR = Path(os.getcwd()).parent
BASE_DIR

In [ ]:
import pandas as pd

fit_df = pd.read_csv(BASE_DIR / DATA_DIR / "resume_job_pairs.csv")

In [ ]:
fit_df

In [ ]:
제목: 건강기능식품
상품개발.주요
업무: 건강기능식품의
기획
및
개발
제조사
선정
핸들링
제품
및
상세페이지
심의
관리
학술자료
리서치
및
분석
제품
관련
부자제
제작
영업사원
제품
교육.자격
요건: 관련
분야
학사
학위
이상
소지자
건강기능식품
개발
경력
3
년
이상
팀워크
및
커뮤니케이션
능력.우대: 영양학, 식품공학 전공자 우대  프로젝트 관리 경험자  외국어 능력 보유자  관련 자격증 소지자.경력: 5~100년.

In [ ]:
서울대학교
컴퓨터공학과
학사
GPA
4.2.경력년수: 0
년.희망직무: 데이터
사이언티스트, 데이터분석가, 데이터엔지니어.기술: Python, SQL, Spark를 활용한 데이터 파이프라인 설계 및 머신러닝 모델링, 가설 검증 기반의 데이터 분석.

In [ ]:
# (1) 패키지 설치

# (2) 스크립트 run_sbert_score.py
import pandas as pd
from sentence_transformers import SentenceTransformer, util
from pathlib import Path
from tqdm.auto import tqdm

IN_CSV = Path("../Data_Files/resume_job_pairs.csv")  # 원본
OUT_CSV = Path("../Data_Files/resume_job_pairs_sbert.csv")  # 결과

df = pd.read_csv(IN_CSV)

model = SentenceTransformer("all-MiniLM-L6-v2")

# 배치 인코딩
resume_emb = model.encode(
    df["resume_text"].tolist(),
    batch_size=32,
    convert_to_tensor=True,
    show_progress_bar=True,
)
job_emb = model.encode(
    df["job_text"].tolist(),
    batch_size=32,
    convert_to_tensor=True,
    show_progress_bar=True,
)

df["score"] = util.cos_sim(resume_emb, job_emb).diagonal().cpu().numpy().round(4)
df.to_csv(OUT_CSV, index=False)
print(f"✅  Saved: {OUT_CSV}")


In [ ]:
df_score = pd.read_csv(OUT_CSV)

In [ ]:
df_score

# 점수 매기기

In [ ]:
import os, json, time
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
from tenacity import retry, wait_random_exponential, stop_after_attempt
from Run_Pipeline.Env_Loader import EnvLoader
import openai

# ── 1. 환경 & 경로 ──────────────────────────────
IN_CSV = Path("../Data_Files/resume_job_pairs.csv")
OUT_CSV = Path("../Data_Files/resume_job_pairs_gpt.csv")
MODEL = "gpt-4o-mini"
BATCH = 4
TEMPERATURE = 0.0

EnvLoader.load_local()  # ⬅️  API 키 env에 주입
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("✅ OPENAI_API_KEY 가 환경변수에 없습니다!")

# v1.x 스타일 클라이언트 (v0.28 이하라면 그냥 openai.ChatCompletion.create 사용)
client = openai.OpenAI(api_key=api_key)

# ── 2. 프롬프트 템플릿 ───────────────────────────
SYSTEM_MSG = """
You are a recruiting expert.
Given a RESUME and a JOB_POST, return a JSON with a suitability score (0.0 to 1.0) and a one-sentence reason.
Score rules:
- 0.9~1.0: almost perfect match
- 0.6~0.8: fairly suitable
- 0.3~0.5: partially suitable
- 0.0~0.2: mostly or completely unsuitable
Respond ONLY in JSON.
"""

USER_TPL = """[RESUME]
{resume}

[JOB_POST]
{job}"""


# ── 3. GPT 호출 (재시도 래퍼) ─────────────────────
@retry(wait=wait_random_exponential(min=1, max=20),
       stop=stop_after_attempt(6))
def gpt_score(resume_text, job_text):
    prompt = USER_TPL.format(resume=resume_text, job=job_text)
    rsp = client.chat.completions.create(
        model=MODEL,
        temperature=TEMPERATURE,
        response_format={"type": "json_object"},  # ★ JSON 강제
        messages=[
            {"role": "system", "content": SYSTEM_MSG},
            {"role": "user", "content": prompt}
        ]
    )
    content = rsp.choices[0].message.content
    return json.loads(content)


# ── 4. CSV 로드 & 라벨링 ─────────────────────────
df = pd.read_csv(IN_CSV)
if "score" not in df.columns:
    df["score"] = ""
    df["reason"] = ""

rows = df.to_dict("records")

for i in tqdm(range(0, len(rows), BATCH)):
    batch = rows[i:i + BATCH]
    for row in batch:
        if row["score"] != "":
            continue
        try:
            res = gpt_score(row["resume_text"], row["job_text"])
            row["score"] = res.get("score", "")
            row["reason"] = res.get("reason", "")
        except Exception as e:
            row["score"] = ""
            row["reason"] = f"ERROR: {e}"

    time.sleep(0.5)  # 속도제한 완충

# ── 5. 저장 ─────────────────────────────────────
pd.DataFrame(rows).to_csv(OUT_CSV, index=False)
print(f"✅ GPT 라벨링 완료 → {OUT_CSV}")


In [ ]:
gpt_df = pd.read_csv(OUT_CSV)

In [ ]:
gpt_df

In [4]:
from Run_Pipeline.Retriever_Builder import RetrieverBuilder
from Run_Pipeline.LLM_Factory import LLMFactory
from Run_Pipeline.Embedding_DB import EmbeddingDB
from Run_Pipeline.Document_Splitter import DocumentSplitter
from Run_Pipeline.Document_Loader import DocumentLoader
from pathlib import Path
import os
from Run_Pipeline.Env_Loader import EnvLoader

EnvLoader.load_local()
BASE_DIR = Path(os.getcwd()).parent

kind = "json"
file_path = BASE_DIR / os.getenv("DATA_PATH", "Data_Files")
chunk_size = 1000
overlap_size = 50
persist_dir = BASE_DIR / os.getenv("DATA_PATH", "Data_Files") / f"{kind}_{chunk_size}"
device = "mps"
retriever_mode = 4
k = 3
engine_num = 1
backend_num = 1

# ③ 문서 로드
loader = DocumentLoader(file_path, kind)  # 일반 파일들을 Document형태로 변환해서 다 불러옴
docs_post, docs_company = loader._load_json_files()

# ④ 문서 분할
splitter = DocumentSplitter(chunk_size=chunk_size, overlap=overlap_size)  # 이미 분할된 문서가 있다면 로드하고 아니면 새로 스플릿함
#chunks_recruitment = splitter.split(docs_post, cache_dir=file_path)
#chunks_company = splitter.split(docs_company, cache_dir=file_path)
chunks = splitter.split(docs_post + docs_company, cache_dir=file_path)  # 게시글과 회사 정보를 합쳐서 분할
# ⑤ 임베딩 & DB 생성/로드
embed_db = EmbeddingDB(model_name="nlpai-lab/KURE-v1", device=device,
                       persist_dir=persist_dir)  # 이미 임베딩된 DB가 있다면 로드하고 아니면 새로 임베딩함
db = embed_db.get_or_create_db(chunks)

# ⑦ LLM 로드
llm = LLMFactory.load_llm(engine=engine_num, backend=backend_num, device=device)  # LLM 로드

# ⑥ 리트리버 빌드
retriever_builder = RetrieverBuilder(
    db=db,
    full_docs=docs_post + docs_company,  # 전체 문서 목록 (게시글 + 회사 정보)
    chunks=chunks,
    k=k,
    mode=retriever_mode
)
retriever = retriever_builder.build()

id = str(283170)
docs = retriever.get_relevant_documents(
    "",  # 쿼리
    search_kwargs={
        "k": 3,
        "filter": {'id': id, 'source': '채용 공고'}  # ← 여기!
    }
)


캐시에서 52358개의 문서 청크 로드 완료
▶ 기존 Chroma DB 로드 중...
▶ 로드 완료


/var/folders/1f/wnkqt1vs7xzfc9mpnzwynpbh0000gn/T/ipykernel_19017/1754827906.py:52: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  docs = retriever.get_relevant_documents(


In [ ]:
docs = retriever.get_relevant_documents(
    "",  # 쿼리
    search_kwargs={
        "k": 3,
        "filter": {'category': '채용 공고'}  # ← 여기!
    }
)

In [5]:
docs

[Document(metadata={'id': '엔젤악기', 'category': '회사 정보'}, page_content='회사명: 엔젤악기\n회사 소개: 설립  1955년판매국가  대한민국, 미국, 프랑스, 영국, 독일, 호주 등 40여개 국가취급품목  악기 및 악세사리브랜드  엔젤악기, ANGEL MUSIC연혁 및 실적1955년 설립19601980년대  서울 본사 확장 이전 경기도 화성시에 공장 설립 국무총리 표창 유망 중소기업으로 선정됨KAIST 선정 대통령 표창1990  2000년대  전국 리코더 콩쿠르 10년 연속 후원 우량 중소기업으로 선정됨경기도 선정 중국 현지 법인 설립 우량 중소기업으로 선정됨경기도 선정 전국 리코더 콩쿠르 20년 연속 후원 수원세무서장 표창납세 의무 성실 이행 청소년위원회 표창 세계 3대 콘서트홀 교육프로그램에서 리코더 선정2010  현재 기업부설연구소 설립 소비자 브랜드 만족도 1위로 선정됨FORBES KOREA 대진대학교와 산학협약을 맺음 경기 문화재단 업무 협연약 스피릿 앙상블 성탄 음악회 협연 충남교육청과 MOU 체결실적 대한민국 최초 PVC 리코더 개발 세계 최초 PVC Frame 글로켄슈필실로폰 개발 세계 최초 리듬 타악기 셋트화가방형 세계 판매 2위 자사 동일 판매 상품기준 세계가 인정한 브랜드자사 브랜드 판매율 95사회공헌 필리핀 케냐 초록우산 전국 리코더 콩쿠르 40년 연속 지원 MIOS\n업종(산업): 제조\n지역: 경기 화성시\n업력: 71년차\n설립년도: 1955\n평균연봉: 3,195만원\n고용보험 가입 사원수: 13명\n국민연금 가입 사원수: 14명\n매출액: 정보 없음\n상세 위치: 경기도 화성시 작현길 46')]

In [ ]:
# 임시로 필터 없이 상위 5개 문서의 메타데이터를 확인합니다.
# "아무 쿼리" 대신, 당신의 데이터와 관련된 일반적인 단어를 넣어봐도 좋습니다.
temp_docs = retriever.get_relevant_documents("채용 공고", search_kwargs={"k": 5}) # 엔젤악기 관련 문서가 있는지 확인하기 위해
print("\n--- 실제 저장된 문서들의 메타데이터 확인 ---")
for doc in temp_docs:
    # 문서 내용의 처음 100자 정도만 출력해서 어떤 문서인지 대략 확인
    print(f"문서 내용: {doc.page_content[:100]}...")
    # 가장 중요한 메타데이터를 출력합니다.
    print(f"메타데이터: {doc.metadata}\n")
print("------------------------------------------\n")